# Measuring Cycle Shapes with Planar Loops

A cycle representative of a 2D image runs along pixel edges. Its staircase is
exact for counting pixels, but it distorts every measurement that involves its
length or its turning: a digital disk has the perimeter of its bounding box,
so its circularity tends to $\pi^2/16 \approx 0.62$ instead of 1, and every
step of the staircase adds a right-angle turn to its curvature.

This tutorial turns representatives into *planar loops* first and measures
those. `persforest.geometry.loops` builds `PlanarLoops`, oriented closed
polylines in physical coordinates, with one function per curve:

| function | grids | point clouds | curve |
| --- | --- | --- | --- |
| `raw_loops(chain, geometry)` | ✓ | ✓ | the staircase of a cubical cycle, the polygon of a simplicial one |
| `midpoint_loops(chain, geometry)` | ✓ | | joins the midpoints of consecutive grid edges |
| `smoothed_loops(chain, geometry, sigma)` | ✓ | ✓ | Gaussian smoothing with arc-length standard deviation `sigma` (physical units) |
| `averaged_loops(chain, geometry, passes=4)` | ✓ | ✓ | `passes` rounds of averaging each vertex with its two neighbours |
| `taut_loops(chain, geometry)` | ✓ | | the shortest closed curve within half a pixel of the staircase |

Smoothed and averaged loops start from the midpoint loop (or the polygon) and
are rescaled to the area of the raw cycle. The functionals of
`persforest.measurements.loops` (`loops_length`, `loops_area`,
`loops_circularity`, `loops_excess_curvature`, convex-hull deficits, ...)
measure `PlanarLoops`; `on_raw_loops`, `on_smoothed_loops`, ... turn them into
`(chain, geometry)` measurements for profiles and landscapes, and
`plot_planar_loops` from `persforest.plotting.loops` draws loops on their own
or on top of a forest plot.

In [ ]:
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np

from persforest import PersistenceForest
from persforest.geometry.loops import averaged_loops, midpoint_loops, raw_loops, smoothed_loops, taut_loops
from persforest.geometry.planar import polygon_area
from persforest.measurements.loops import (
    loops_area, loops_circularity, loops_excess_curvature, loops_length,
    on_averaged_loops, on_raw_loops, on_smoothed_loops, on_taut_loops,
)
from persforest.plotting.loops import plot_planar_loops

SPACING = 0.05  # physical size of a pixel

# The curves compared throughout, as (chain, geometry) -> PlanarLoops.
CURVES = {
    "raw": raw_loops,
    "midpoint": midpoint_loops,
    "smoothed": lambda chain, geometry: smoothed_loops(chain, geometry, sigma=2 * SPACING),
    "averaged": averaged_loops,
    "taut": taut_loops,
}


def peaks_image(shape=(150, 100)):
    # Three peaks on a flat background, built with (axis 0, axis 1) coordinates.
    # Every level set of a peak is the same shape, scaled: a disk, a 2:1
    # ellipse rotated by 30 degrees and a square rotated by 20 degrees.
    x, y = (np.mgrid[0:shape[0], 0:shape[1]] + 0.5) * SPACING

    def rotated(center, degrees):
        c, s = math.cos(math.radians(degrees)), math.sin(math.radians(degrees))
        return c * (x - center[0]) + s * (y - center[1]), -s * (x - center[0]) + c * (y - center[1])

    u, v = rotated((1.3, 2.5), 0)
    disk = 1 - np.hypot(u, v) / 1.1
    u, v = rotated((3.75, 1.45), 30)
    ellipse = 1 - np.hypot(u / 1.1, v / 0.55)
    u, v = rotated((5.9, 3.1), 20)
    square = 1 - np.maximum(abs(u), abs(v)) / 0.9
    return np.maximum.reduce([disk, ellipse, square, np.zeros(shape)])


def ellipse_circularity(a, b):
    # Ramanujan's perimeter approximation; accurate to 1e-5 for a 2:1 ellipse.
    perimeter = math.pi * (3 * (a + b) - math.sqrt((3 * a + b) * (a + 3 * b)))
    return 4 * math.pi * math.pi * a * b / perimeter**2


def distance_to_staircase(loops, staircase):
    # Largest distance of a loop vertex from the staircase, in pixels.
    start = np.concatenate(staircase.loops)
    end = np.concatenate([np.roll(loop, -1, axis=0) for loop in staircase])
    step = end - start
    points = loops.points
    t = np.clip(((points[:, None] - start) * step).sum(2) / (step**2).sum(1), 0, 1)
    return np.linalg.norm(points[:, None] - (start + t[..., None] * step), axis=2).min(1).max() / SPACING


PEAKS = {  # center and true circularity of every level set
    "disk": ((1.3, 2.5), 1.0),
    "ellipse": ((3.75, 1.45), ellipse_circularity(2, 1)),
    "square": ((5.9, 3.1), math.pi / 4),
}

## A forest with three known shapes

In the sublevel filtration the background enters at 0, closing a loop around
every peak; the loop shrinks onto the peak as the filtration value grows. All
level sets of a peak have the same shape, so a faithful circularity stays
constant along each bar.

In [ ]:
image = peaks_image()
forest = PersistenceForest.from_cubical_grid(image, spacing=SPACING, storage="cycles")


def peak_of(bar, value=0.3):
    center = raw_loops(forest.cycle_for_bar_at(bar, value), forest.geometry).points.mean(axis=0)
    return min(PEAKS, key=lambda name: np.hypot(*(np.asarray(PEAKS[name][0]) - center)))


bars = {peak_of(bar): bar for bar in forest.longest_bars(3)}
for name, bar in bars.items():
    print(f"{name:8s} bar [{bar.birth:.3f}, {bar.death:.3f})")

T = 0.3
forest.plot_at_filtration(T, figsize=(7, 5), show=False, title=f"representatives at {T}")
plt.show()

## From a representative to loops

Each function returns `PlanarLoops`; `plot_planar_loops` draws them on top of
`plot_at_filtration` with `show_cycles=False`. Here are the five curves of the
ellipse's representative; the smoothed loop uses `sigma = 2 * SPACING`, two
pixels. The last panel zooms into the lower end of the ellipse and draws the
three curves meant to follow the shape on top of the staircase. The domain
outline is switched off because the panels are zoomed in.

In [ ]:
ZOOM_STYLE = {"domain_edge_width": 0, "image_colorbar": False}
COLORS = {"raw": "tab:blue", "midpoint": "tab:orange", "smoothed": "crimson", "averaged": "tab:purple",
          "taut": "tab:green"}


def zoom_to(ax, points, margin=0.1):
    lower, upper = points.min(axis=0) - margin, points.max(axis=0) + margin
    ax.set_xlim(lower[0], upper[0])
    ax.set_ylim(lower[1], upper[1])


rep = forest.cycle_for_bar_at(bars["ellipse"], T)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (name, build) in zip(axes.flat, CURVES.items()):
    loops = build(rep, forest.geometry)
    forest.plot_at_filtration(T, ax=ax, show=False, show_cycles=False, style_2d=ZOOM_STYLE, title=name)
    plot_planar_loops(loops, ax=ax, show=False, color=COLORS[name], show_vertices=name != "smoothed")
    zoom_to(ax, loops.points)

ax = axes.flat[-1]
staircase = raw_loops(rep, forest.geometry)
forest.plot_at_filtration(T, ax=ax, show=False, show_cycles=False, style_2d=ZOOM_STYLE, title="close-up")
plot_planar_loops(staircase, ax=ax, show=False, color=COLORS["raw"], linewidth=1.5, label="raw")
for name in ("smoothed", "averaged", "taut"):
    plot_planar_loops(CURVES[name](rep, forest.geometry), ax=ax, show=False, color=COLORS[name], label=name)
lowest = staircase.points[staircase.points[:, 0] < staircase.points[:, 0].min() + 0.5]
zoom_to(ax, lowest, margin=0.05)
ax.legend(loc="lower right", framealpha=0.9)
fig.tight_layout()
plt.show()

for name, build in CURVES.items():
    loops = build(rep, forest.geometry)
    print(f"{name:9s} kind={loops.kind!r:12s} params={dict(loops.params)}  {len(loops.loops[0]):4d} vertices")

## Three ways to follow the shape

The staircase is itself up to about half a pixel away from the boundary it
approximates. A curve that stays much closer than that keeps its zigzag, so
every reconstruction needs some room:

- **smoothed** filters the whole loop with a Gaussian of width `sigma` along
  its length. It removes the steps completely, but its reach grows with
  `sigma`, and corners round off over that length.
- **averaged** moves every vertex of the midpoint loop toward its two
  neighbours, `passes` times. Each pass acts only locally, so with four
  passes the curve stays within about 0.2 to 0.4 pixels of the staircase. It
  keeps a little of the step pattern.
- **taut** is the shortest closed curve that crosses every staircase edge on
  the segment between the centers of the two pixels it separates. Its corners
  are pixel centers, it touches the band of half a pixel around the staircase
  at every corner, and it has no parameter. A convex region gives a convex
  polygon, so its excess curvature is zero. It is not rescaled: it encloses
  slightly less than the pixel area, and a region one pixel wide has no area.

The largest distance of each curve from the staircase, in pixels:

In [ ]:
print(f"{'':8s}" + "".join(f"{name:>10s}" for name in ("midpoint", "smoothed", "averaged", "taut")))
for name, bar in bars.items():
    rep = forest.cycle_for_bar_at(bar, T)
    staircase = raw_loops(rep, forest.geometry)
    distances = [distance_to_staircase(CURVES[curve](rep, forest.geometry), staircase)
                 for curve in ("midpoint", "smoothed", "averaged", "taut")]
    print(f"{name:8s}" + "".join(f"{distance:10.2f}" for distance in distances))

## What the curves measure

Circularity $4\pi A / L^2$ and excess curvature (total absolute turning minus
$2\pi$, zero for every convex curve) of each representative at value 0.3,
next to the true circularity of its level sets.

- The **staircase** puts every shape near 0.5 and cannot tell the square
  from the ellipse; its curvature counts the turns of the grid.
- The **midpoint** loop recovers most of the length (a few percent too long)
  but still turns at every step.
- **Smoothed**, **averaged** and **taut** loops come within a few hundredths
  of the true circularity of the disk and the ellipse. All three make the
  square somewhat rounder than it is, because each one cuts its corners.
- Only the taut loop has no excess curvature on every shape; on the square
  the averaged loop keeps some turning from the steps, the smoothed loop a
  little.

Excess curvature of staircase and midpoint loops emits a `UserWarning` for
exactly this reason; the table silences it.

In [ ]:
print(f"{'':8s} {'true':>6s} |" + "".join(f"{name:>9s}" for name in CURVES) + "  | excess curvature (rad)")
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    for name, bar in bars.items():
        rep = forest.cycle_for_bar_at(bar, T)
        loops = [build(rep, forest.geometry) for build in CURVES.values()]
        print(f"{name:8s} {PEAKS[name][1]:6.3f} |"
              + "".join(f"{loops_circularity(l):9.3f}" for l in loops) + "  |"
              + "".join(f"{loops_excess_curvature(l):7.2f}" for l in loops))

rep = forest.cycle_for_bar_at(bars["disk"], T)
print(f"\ndisk at {T}: {len(rep.signed_simplices)} grid edges")
for name, build in CURVES.items():
    loops = build(rep, forest.geometry)
    print(f"  {name:9s} length {loops_length(loops):.4f}  area {loops_area(loops):.4f}")

The area column shows the bookkeeping: the staircase encloses the pixel area
exactly, the midpoint loop loses half a pixel ($\tfrac12$ `spacing[0] * spacing[1]`)
per loop, the smoothed and averaged loops are rescaled to the pixel area, and
the taut loop encloses its own, slightly smaller area.

## Choosing sigma and passes

Both parameters are modelling choices. Small values leave pixel steps in the
curve; large values round corners and pull the curve away from the cycle.
For `sigma`, two pixels removes the steps but already moves the curve almost a
pixel away at the corners; `passes=4` keeps it within about 0.4 pixels. The
taut loop (dashed) needs no choice and stays within half a pixel. The square
shows the trade-off best:

In [ ]:
rep = forest.cycle_for_bar_at(bars["square"], T)
staircase = raw_loops(rep, forest.geometry)
sweeps = {
    "sigma": [(f"{factor:g} px", smoothed_loops(rep, forest.geometry, factor * SPACING)) for factor in (0.5, 1, 2, 4, 8)],
    "passes": [(f"{passes}", averaged_loops(rep, forest.geometry, passes)) for passes in (1, 2, 4, 8, 16)],
}
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (parameter, variants) in zip(axes, sweeps.items()):
    forest.plot_at_filtration(T, ax=ax, show=False, show_cycles=False, style_2d=ZOOM_STYLE,
                              title=f"square: increasing {parameter}")
    colors = plt.cm.viridis(np.linspace(0, 0.9, len(variants)))
    print(parameter)
    for (label, loops), color in zip(variants, colors):
        plot_planar_loops(loops, ax=ax, show=False, color=color, linewidth=1.5, label=label)
        print(f"  {label:6s}: circularity {loops_circularity(loops):.3f}, "
              f"excess curvature {loops_excess_curvature(loops):5.2f}, "
              f"distance to staircase {distance_to_staircase(loops, staircase):.2f} px")
    taut = taut_loops(rep, forest.geometry)
    plot_planar_loops(taut, ax=ax, show=False, color="tab:red", linewidth=1, linestyle="--", label="taut")
    zoom_to(ax, staircase.points)
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), title=parameter)
print(f"taut: circularity {loops_circularity(taut):.3f}, true {math.pi / 4:.3f}")
fig.tight_layout()
plt.show()

## Small cycles

On a grid a cycle cannot be shorter than the four edges of one pixel. The
curves treat such cycles very differently:

- the midpoint loop of a pixel is a diamond with perimeter $2\sqrt{2}$;
- smoothing turns it into the circle of the same area, perimeter $2\sqrt{\pi}$;
- averaging keeps a square of the same area, since averaging leaves a
  diamond a diamond;
- the taut loop collapses onto the pixel center: a single vertex with no
  length and no area. A 1×3 bar becomes the segment through its pixel
  centers, traversed there and back.

Circularity of a loop without length is zero by convention.

In [ ]:
small = np.zeros((12, 8))
small[3, 3] = 1      # one pixel
small[6:9, 3] = 1    # a 1x3 bar along axis 0
small_forest = PersistenceForest.from_cubical_grid(small, storage="cycles")
small_curves = {**CURVES, "smoothed": lambda chain, geometry: smoothed_loops(chain, geometry, sigma=1.0)}

fig, ax = plt.subplots(figsize=(8, 3.2))
small_forest.plot_at_filtration(0.5, ax=ax, show=False, show_cycles=False, title="pixel and 1x3 bar",
                                style_2d={"image_colorbar": False})
small_bars = sorted((bar for bar in small_forest.barcode if bar.birth <= 0.5 < bar.death), key=lambda bar: bar.birth)
for index, bar in enumerate(small_bars):
    rep = small_forest.cycle_for_bar_at(bar, 0.5)
    for name, build in small_curves.items():
        loops = build(rep, small_forest.geometry)
        plot_planar_loops(loops, ax=ax, show=False, color=COLORS[name], label=name if index == 0 else None,
                          show_vertices=name == "taut")
        print(f"{len(rep.signed_simplices)} edges, {name:9s}: {len(loops.loops[0]):3d} vertices, "
              f"length {loops_length(loops):.4f}, area {loops_area(loops):.3f}, "
              f"circularity {loops_circularity(loops):.3f}")
ax.set_xlim(2.2, 9.8)
ax.set_ylim(2.2, 4.8)
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
plt.show()

## Holes and orientation

A representative bounds one connected component of the complement of the
complex. When that component has holes (islands of the complex inside it),
its boundary has several loops: the outer one runs counterclockwise
(positive signed area) and each hole boundary clockwise (negative signed
area). `loops_area` sums the signed areas, which is the outer area minus the
hole areas. Here a bright ring encloses a dark island. Pass
`show_orientation=True` to `plot_planar_loops` to mark the direction of each
loop.

In [ ]:
x, y = (np.mgrid[0:60, 0:60] + 0.5) * SPACING - 1.5
radius = np.hypot(x, 1.4 * y)
ring = ((radius < 1.2) & (radius > 0.45)).astype(float)
ring_forest = PersistenceForest.from_cubical_grid(ring, spacing=SPACING, storage="cycles")
[ring_bar] = [bar for bar in ring_forest.barcode if bar.death - bar.birth > 0.5]
rep = ring_forest.cycle_for_bar_at(ring_bar, 0.5)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, name in zip(axes, ("raw", "smoothed", "taut")):
    loops = CURVES[name](rep, ring_forest.geometry)
    ring_forest.plot_at_filtration(0.5, ax=ax, show=False, show_cycles=False, title=name,
                                   style_2d={"image_colorbar": False})
    plot_planar_loops(loops, ax=ax, show=False, color="crimson", show_orientation=False, linewidth=2)
    signed = [polygon_area(loop, signed=True) for loop in loops]
    print(f"{name:8s}: signed loop areas {np.round(signed, 4)} -> loops_area {loops_area(loops):.4f}")
print(f"pixel count x pixel area: {ring.sum() * SPACING**2:.4f}")
plt.show()

## Measurement profiles and landscapes

`on_raw_loops(functional)`, `on_smoothed_loops(functional, sigma)`,
`on_averaged_loops(functional, passes)`, `on_taut_loops(functional)` (and
`on_midpoint_loops`) turn a loop functional into the `(chain, geometry)`
measurement that profiles and landscapes expect. Further keyword arguments go
to the functional. The returned function carries a `measurement_name` and
`measurement_params`, which the profiles record.

Profiles use the bar colors of the first figure (disk orange, ellipse blue,
square green); dashed lines mark the true circularities.

In [ ]:
measurements = [on_raw_loops(loops_circularity),
                on_smoothed_loops(loops_circularity, sigma=2 * SPACING),
                on_averaged_loops(loops_circularity, passes=4),
                on_taut_loops(loops_circularity)]
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharey=True)
for ax, measurement in zip(axes.flat, measurements):
    profiles = forest.compute_measurement_profiles(measurement, measurement.measurement_name, min_bar_length=0.5)
    print(f"{profiles.measurement_name:45s} {dict(profiles.measurement_params)}")
    forest.plot_measurement_profiles(profiles, ax=ax, min_bar_length=0.5, title=profiles.measurement_name,
                                     ylabel="circularity")
    for _, circularity in PEAKS.values():
        ax.axhline(circularity, color="0.5", linewidth=0.8, linestyle="--")
    ax.set_ylim(0, 1.05)
fig.tight_layout()
plt.show()

- **raw** profiles saw-tooth with every added pixel and do not separate the
  square from the ellipse.
- **smoothed** profiles stay at their true values for the disk and the
  ellipse. The square drifts toward 1 as it shrinks: the fixed two-pixel
  `sigma` rounds an ever larger part of it.
- **averaged** profiles behave alike, with a little more noise and less
  drift of the square, since four passes reach less far.
- **taut** profiles fluctuate more, because the polygon gains and loses
  corners one pixel at a time, and they read slightly low for round shapes,
  whose polygon cuts the curve. At the end of a bar, when the region is only a
  pixel or two wide, the polygon collapses and the circularity drops to zero.

Landscapes take the same measurements.

In [ ]:
taut_circularity = measurements[3]
family = forest.compute_measurement_landscapes(taut_circularity, "taut circularity", max_k=3, min_bar_length=0.5)
forest.plot_measurement_landscapes("taut circularity")
plt.show()

## Point clouds

`smoothed_loops` and `averaged_loops` also apply to cycles of point clouds;
they work on the polygon of the representative (there is no midpoint or taut
curve). `sigma` is in the units of the points. `passes` counts polygon
vertices, which can be unevenly spaced, so the reach of averaging varies along
the loop. Below, an alpha complex of a noisy 2:1 ellipse shortly after its
loop closes: the raw polygon zig-zags between sample points, which both
reconstructions remove while keeping the enclosed area. The representative
runs along the inner side of the noisy band, so it encloses less than the
ellipse through the sample centers.

In [ ]:
rng = np.random.default_rng(0)
angles = rng.uniform(0, 2 * np.pi, 250)
points = np.column_stack([2 * np.cos(angles), np.sin(angles)]) + rng.normal(scale=0.06, size=(250, 2))
cloud_forest = PersistenceForest.from_points(points, storage="cycles")
bar = cloud_forest.max_bar()
value = bar.birth + 0.05 * (bar.death - bar.birth)
rep = cloud_forest.cycle_for_bar_at(bar, value)

cloud_curves = {"raw": raw_loops(rep, cloud_forest.geometry),
                "smoothed": smoothed_loops(rep, cloud_forest.geometry, sigma=0.15),
                "averaged": averaged_loops(rep, cloud_forest.geometry, passes=4)}
fig, ax = plt.subplots(figsize=(8, 4.5))
cloud_forest.plot_at_filtration(value, ax=ax, show=False, show_cycles=False, title="alpha complex of a noisy ellipse")
for name, loops in cloud_curves.items():
    plot_planar_loops(loops, ax=ax, show=False, color=COLORS[name], linewidth=1.8, label=name)
    print(f"{name:8s}: circularity {loops_circularity(loops):.3f}, area {loops_area(loops):.3f}")
print(f"ellipse 2:1: circularity {ellipse_circularity(2, 1):.3f}")
ax.legend(loc="upper right")
plt.show()

## Summary

| step | function |
| --- | --- |
| representative → loops | `raw_loops`, `midpoint_loops`, `smoothed_loops(..., sigma)`, `averaged_loops(..., passes)`, `taut_loops` |
| measure loops | `loops_length`, `loops_area`, `loops_circularity` (and complements), `loops_non_circularity`, `loops_convex_hull_area_deficit`, `loops_convex_hull_perimeter_deficit`, `loops_max_convexity_defect_depth`, `loops_excess_curvature` |
| profiles and landscapes | `on_raw_loops(functional, **kwargs)`, `on_midpoint_loops`, `on_smoothed_loops(functional, sigma)`, `on_averaged_loops(functional, passes)`, `on_taut_loops` |
| plot | `plot_planar_loops(loops, ax=..., image_orientation=...)` |

All reconstructions are built from `chain.unsigned()`: they describe the
region the cycle bounds. Tendrils, which appear on grids with
`filtration_type="vertices"`, are measured on the chain itself with
`cubical_chain_avg_tendril_length`, `cubical_chain_num_of_branching_points`
and the other `cubical_chain_*` functionals of `persforest.measurements.cubical`.
When plotting with `image_orientation=True`, pass the same flag to
`plot_planar_loops`.